In [1]:
%pip install -q -U sentence-transformers

%pip install -q supabase datasets tqdm torch

In [1]:
import os

os.environ["HF_TOKEN"] = input()

In [2]:
SUPABASE_URL = "https://mfphsrdubggjqxvyuzil.supabase.co"

SUPABASE_SERVICE_ROLE_KEY = input()

In [3]:
from datasets import load_dataset, Dataset

ds = load_dataset("pameydorke/arcanum-cross-platform-queries-synthetic")

README.md:   0%|          | 0.00/585 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.93MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  743kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4612 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1153 [00:00<?, ? examples/s]

In [4]:
import pandas as pd

traind_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()
full_df = pd.concat([traind_df, test_df])

unique_docs_df = full_df.drop_duplicates(subset=['document_text'])
unique_docs_df = unique_docs_df.reset_index()
unique_docs_df = unique_docs_df.drop(columns=['index', 'query', 'query_type', 'difficulty'])

unique_docs_ds = Dataset.from_pandas(unique_docs_df)

unique_docs_ds

Dataset({
    features: ['document_title', 'document_text', 'document_link'],
    num_rows: 1153
})

In [5]:
from supabase import create_client

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("pameydorke/arcanumchat-retriever-bge-m3-v2")

modules.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/284 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/42.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

In [7]:
from tqdm.auto import tqdm
import numpy as np

def update_embeddings_in_batches(dataset, batch_size=32):
    for i in tqdm(range(0, len(dataset), batch_size), desc="Batches"):
        batch = dataset[i : i + batch_size]
        texts = batch["document_text"]

        embeddings = model.encode(
            texts,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).tolist()

        for text, emb in zip(texts, embeddings):
            response = (
                supabase.table("documents")
                .update({"embedding": emb})
                .eq("content", text)
                .execute()
            )
            if not response:
                print(text[:200])

In [8]:
update_embeddings_in_batches(unique_docs_ds, batch_size=32)

Batches:   0%|          | 0/37 [00:00<?, ?it/s]

In [9]:
import json

sample_text = unique_docs_ds[42]["document_text"]

row = supabase.table("documents").select("id, content, embedding").eq("content", sample_text).single().execute()

emb = json.loads(row.data["embedding"])

assert len(emb) == 1024

print(emb[:5])

[-0.034524854, 0.015796266, 0.044990912, 0.00037731856, 0.045016542]


In [10]:
print(model.encode(
    [sample_text],
    batch_size=1,
    normalize_embeddings=True,
    show_progress_bar=False,
).tolist()[0][:5])

[-0.03452487289905548, 0.01579628325998783, 0.04499088600277901, 0.0003772749041672796, 0.04501654580235481]
